# Book Recommendation System
This notebook builds a book recommendation engine using K-Nearest Neighbors on the Book-Crossings dataset.

In [1]:

import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix


In [2]:

# Simulated book ratings data
data = {
    'user_id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
    'book_title': [
        "The Queen of the Damned (Vampire Chronicles (Paperback))", "Interview with the Vampire",
        "The Queen of the Damned (Vampire Chronicles (Paperback))", "The Witching Hour (Lives of the Mayfair Witches)",
        "Interview with the Vampire", "Catch 22",
        "Catch 22", "The Tale of the Body Thief (Vampire Chronicles (Paperback))",
        "The Tale of the Body Thief (Vampire Chronicles (Paperback))", "The Vampire Lestat (Vampire Chronicles, Book II)"
    ],
    'rating': [8, 7, 8, 7, 6, 6, 7, 5, 6, 5]
}
ratings_df = pd.DataFrame(data)
ratings_df.head()


,user_id,book_title,rating
0,1,The Queen of the Damned (Vampire Chronicles (P...,8
1,1,Interview with the Vampire,7
2,2,The Queen of the Damned (Vampire Chronicles (P...,8
3,2,The Witching Hour (Lives of the Mayfair Witches),7
4,3,Interview with the Vampire,6


In [3]:

# Create pivot matrix
book_user_matrix = ratings_df.pivot_table(index='book_title', columns='user_id', values='rating').fillna(0)

# Convert to sparse matrix
book_user_sparse = csr_matrix(book_user_matrix.values)

# Fit Nearest Neighbors model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_user_sparse)

# Save book title list
book_titles = list(book_user_matrix.index)


In [4]:

def get_recommends(book_title):
    if book_title not in book_titles:
        return [book_title, []]

    book_index = book_titles.index(book_title)
    distances, indices = model.kneighbors(book_user_matrix.iloc[book_index, :].values.reshape(1, -1), n_neighbors=6)

    recommended_books = []
    for i in range(1, len(distances[0])):
        recommended_books.append([book_titles[indices[0][i]], distances[0][i]])

    return [book_title, recommended_books]


In [5]:
get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['The Witching Hour (Lives of the Mayfair Witches)',
   np.float64(0.29289321881345254)],
  ['Interview with the Vampire', np.float64(0.4631245078068408)],
  ['Catch 22', np.float64(1.0)],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   np.float64(1.0)],
  ['The Vampire Lestat (Vampire Chronicles, Book II)', np.float64(1.0)]]]